# Web Scraping — E-commerce Product Data

A web scraper built with **requests** and **BeautifulSoup** that extracts structured product data from an online store, collects the results of a search into a table, and exports a filtered subset to Excel.

**Techniques:** HTTP requests, HTML parsing (BeautifulSoup / lxml), extracting fields from the DOM, iterating over search-result pages, building a pandas DataFrame, exporting to Excel.

### 1. Imports

In [ ]:
import requests
import pandas as pd
import bs4


### 2. Scrape a single product page
A function that, given a product URL, extracts the title, a short description, the price, and whether the product has selectable variants.

In [ ]:
def scrape_product_details(url):
  p=requests.get(url)
  page = bs4.BeautifulSoup(p.content,'lxml')
  title = page.h1.text
  product_description=page.find("div",class_="rte").text.strip().replace("\n"," ")
  if len(product_description)>100:
    product_description=product_description[0:99]
  product_price=float(page.find("div",class_="desc_blk_bot clearfix").find("span",class_="money").text.strip().split("$")[1])
  weightable=False
  if page.find("div",class_="variations pl10").find("div",class_="selector-wrapper main-product-select") is not None:
    weightable=True

  result=(title,product_description,product_price,weightable)
  return(result)

Quick test on two product pages:

In [ ]:
scrape_product_details("https://americanmadestore.us/collections/food/products/american-made-jam-apricot-1-pt")
scrape_product_details("https://americanmadestore.us/products/gummi-bears?_pos=10&_sid=fb3afa9fb&_ss=r")

('Gummi Bears',
 'Six classic flavors are packed into this bear mix. Strawberry, apple, orange, pineapple, lemon and ',
 3.95,
 True)

### 3. Scrape a full search-results page
We fetch a search page, find every product listing, follow each link and scrape its details, then assemble everything into a pandas DataFrame.

In [ ]:
url= "https://americanmadestore.us/search?type=product&q=apple"
p=requests.get(url)
page=bs4.BeautifulSoup(p.content,'lxml')
ads_tot=page.find_all("div",class_="main_box")
results=[]
for i in ads_tot:
  ad_url_fin=str(i.a["href"])
  ad_url="https://americanmadestore.us"+ad_url_fin
  results.append(scrape_product_details(ad_url))
final_file=pd.DataFrame(results, columns=["Title", "Description", "Price" ,"Weightable"])
print(final_file)

                                                Title  \
0                 Amish Wedding Apple Butter (32 oz.)   
1               Claeys Sanded Green Apple Drops- 6oz.   
2            Amish Wedding Apple Pie Filling (32 oz.)   
3     Amish Wedding Old Fashioned Apple Butter (16oz)   
4        Amish Wedding Sugar Free Apple Butter (16oz)   
5   Milkhouse Candle Co. Caramel Apple Fragrance Melt   
6                Alice's Cottage Spiced Mug Mat Apple   
7                        Arkansalsa Small Batch Salsa   
8              Amish Wedding Old Fashioned Mild Salsa   
9                                         Gummi Bears   
10                                   Benica's Shampoo   
11                             Gummi Rainforest Frogs   

                                          Description  Price  Weightable  
0   Old-fashioned, all-natural apple butter For pr...  10.99       False  
1   These tart yet sweet green apple drops are cov...   1.99       False  
2   For great tasting pies, add A

### 4. Filter and export to Excel
A helper that keeps only the products below a given price and saves them to an `.xlsx` file.

In [ ]:
def filter_by_price(price):
  data=final_file
  data=data[data["Price"]<price]
  data.to_excel(f"Filtered_dataframe_{price}.xlsx")

In [ ]:
filter_by_price(8)